## Importación de los datos

In [5]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from scipy import stats
import os

In [13]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

dfs_pm25 = []

for res in resources_PM25:
    es_csv = "CSV" in res["format"].upper() # Anteriormente usé "res["format"].upper() == "CSV"" pero esto excluia los CSV del 2014, 2022, 2023 porque tienen terminación .CSV 
    es_metadato = "aire-" in res["url"]
    es_historico_manual = "manuales" in res["url"]

    if es_csv and not es_metadato and not es_historico_manual:
        try:
            df_temp = pd.read_csv(res["url"], storage_options=headers)
            if "pollutant_id" in df_temp.columns:
                df_pm2 = df_temp[df_temp["pollutant_id"] == "PM2"].copy()
                df_pm2["archivo_origen"] = res["name"]
                dfs_pm25.append(df_pm2)
                print(f"{res['name']}: {len(df_pm2)} filas de PM2 (de {len(df_temp)} totales)")
        except Exception as e:
            print(f"Error con {res['name']}: {e}")

pm25_completo = pd.concat(dfs_pm25, ignore_index=True)
print("\nTotal filas de PM2:", len(pm25_completo)) # Esto es más una formalidad que otra cosa
pm25_completo.head()

Medidas de la calidad del aire - 2014: 0 filas de PM2 (de 13422 totales)
Medidas de la calidad del aire - 2015: 8688 filas de PM2 (de 25838 totales)
Medidas de la calidad del aire - 2016: 8784 filas de PM2 (de 18377 totales)
Medidas de la calidad del aire - 2017: 18448 filas de PM2 (de 49383 totales)
Medidas de la calidad del aire - 2018: 26244 filas de PM2 (de 71121 totales)
Medidas de la calidad del aire - 2019: 23028 filas de PM2 (de 69936 totales)
Medidas de la calidad del aire - 2020: 26520 filas de PM2 (de 79056 totales)
Medidas de la calidad del aire - 2021: 26280 filas de PM2 (de 70080 totales)
Medidas de la calidad del aire - 2022: 26280 filas de PM2 (de 70080 totales)
Medidas de la calidad del aire - 2023: 26280 filas de PM2 (de 70080 totales)
Medidas de la calidad del aire - 2024: 26352 filas de PM2 (de 70272 totales)
Medidas de la calidad del aire – 2025: 26280 filas de PM2 (de 61320 totales)

Total filas de PM2: 243184


,pollutant_id,pollutant_averaging,date,pollutant_value,pollutant_unit,station_id,X,Y,ID_estacion,method_id,archivo_origen,Unnamed: 0
0,PM2,1,2015-01-04 00:00:00,8.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN
1,PM2,1,2015-01-04 01:00:00,7.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN
2,PM2,1,2015-01-04 02:00:00,6.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN
3,PM2,1,2015-01-04 03:00:00,4.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN
4,PM2,1,2015-01-04 04:00:00,4.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN


## Visualización primaria de los datos

In [14]:
print(pm25_completo['archivo_origen'].value_counts())

archivo_origen
Medidas de la calidad del aire - 2020    26520
Medidas de la calidad del aire - 2024    26352
Medidas de la calidad del aire - 2021    26280
Medidas de la calidad del aire - 2022    26280
Medidas de la calidad del aire - 2023    26280
Medidas de la calidad del aire – 2025    26280
Medidas de la calidad del aire - 2018    26244
Medidas de la calidad del aire - 2019    23028
Medidas de la calidad del aire - 2017    18448
Medidas de la calidad del aire - 2016     8784
Medidas de la calidad del aire - 2015     8688
Name: count, dtype: int64


In [22]:
pm25_completo.info()
print("-----------------------------------")
print(f"El shape es: {pm25_completo.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243184 entries, 0 to 243183
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   pollutant_id         243184 non-null  object 
 1   pollutant_averaging  243184 non-null  int64  
 2   date                 243184 non-null  object 
 3   pollutant_value      213961 non-null  float64
 4   pollutant_unit       243184 non-null  object 
 5   station_id           243184 non-null  object 
 6   X                    243184 non-null  int64  
 7   Y                    243184 non-null  int64  
 8   ID_estacion          243184 non-null  object 
 9   method_id            243184 non-null  object 
 10  archivo_origen       243184 non-null  object 
 11  Unnamed: 0           75828 non-null   float64
dtypes: float64(2), int64(3), object(7)
memory usage: 22.3+ MB
-----------------------------------
El shape es: (243184, 12)


In [26]:
pm25_completo.head(1)

,pollutant_id,pollutant_averaging,date,pollutant_value,pollutant_unit,station_id,X,Y,ID_estacion,method_id,archivo_origen,Unnamed: 0
0,PM2,1,2015-01-04 00:00:00,8.0,ug/m3,UYMVD_E1,572796,6137122,Ciudad Vieja 3,UYMVD_PM2_b,Medidas de la calidad del aire - 2015,NaN


In [27]:
pm25_completo.tail(1)

,pollutant_id,pollutant_averaging,date,pollutant_value,pollutant_unit,station_id,X,Y,ID_estacion,method_id,archivo_origen,Unnamed: 0
243183,PM2,1,2025-12-31 23:00:00,27.0,ug/m3,UYMVD_E1,572452,6137044,Ciudad Vieja2,UYMVD_PM2_b,Medidas de la calidad del aire – 2025,NaN


In [28]:
pm25_completo.isna().sum()

pollutant_id                0
pollutant_averaging         0
date                        0
pollutant_value         29223
pollutant_unit              0
station_id                  0
X                           0
Y                           0
ID_estacion                 0
method_id                   0
archivo_origen              0
Unnamed: 0             167356
dtype: int64

In [33]:
# % de nulos por estación
print(pm25_completo.groupby("ID_estacion")['pollutant_value'].apply(lambda x: x.isna().mean() * 100))

ID_estacion
Ciudad Vieja 3       2.452999
Ciudad Vieja2        4.963284
Ciudad Vieja3        9.376620
Colon                3.264537
Curva de Maronas    14.753641
Tres Cruces 3       49.224884
Tres Cruces 4       14.746171
Name: pollutant_value, dtype: float64


In [38]:
# Cantidad de estaciones (ID-estacion)
print(f"Estaciones separadas por: {pm25_completo["ID_estacion"].value_counts()}")

Estaciones separadas por: ID_estacion
Curva de Maronas    78889
Tres Cruces 4       48182
Ciudad Vieja 3      34733
Ciudad Vieja3       30864
Ciudad Vieja2       30504
Colon               10078
Tres Cruces 3        9934
Name: count, dtype: int64


In [39]:
# Cantidad de estaciones con identificadores únicos (station_id)
print(f"Estaciones separadas por: {pm25_completo["station_id"].value_counts()}")

Estaciones separadas por: station_id
UYMVD_E1    96101
UYMVD_E6    78889
UYMVD_E5    58116
UYMVD_E8    10078
Name: count, dtype: int64


In [47]:
coordenadas_de_estaciones = pm25_completo[["ID_estacion", "X", "Y"]].drop_duplicates()
coordenadas_de_estaciones

,ID_estacion,X,Y
0,Ciudad Vieja 3,572796,6137122
17472,Curva de Maronas,579229,6142255
26065,Colon,570970,6149046
27419,Ciudad Vieja 3,572796,6137123
35920,Curva de Maronas,579230,6142255
70924,Tres Cruces 3,576247,6138473
76432,Ciudad Vieja3,572796,6137123
98570,Tres Cruces 4,576324,6138361
160072,Ciudad Vieja2,572452,6137044
